<a href="https://colab.research.google.com/github/venezianof/booksum/blob/main/gpt_ossmio030626pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Installation & Environment Setup
We begin by installing the Unsloth library and its dependencies, which allow for memory-efficient training of large models on consumer GPUs like the Tesla T4.

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-prttkhvi/unsloth_4e14c134f42149049c7477e88b5adda0
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-prttkhvi/unsloth_4e14c134f42149049c7477e88b5adda0
  Resolved https://github.com/unslothai/unsloth.git to commit b0572bd233efc120d50a940f13af89eb879d0dd0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 110.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 9.7 MB/s eta 0:00:00


### 2. Model Configuration
We load the GPT-OSS 20B model in 4-bit quantization to fit within the T4's 16GB VRAM and apply LoRA adapters.

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 128,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [4]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
load_in_4bit = True

# Load the model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 128,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("Model successfully loaded on GPU!")

NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

### 3. Reward Functions
We define the logic to evaluate the model's output. This includes checking for syntax, ensuring no external libraries like NumPy are imported (anti-cheating), and measuring execution speed relative to a baseline.

In [3]:
import ast
import time
import numpy as np

def correctness_reward(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        try:
            # Basic validation of syntax and logic
            exec_globals = {}
            exec(completion, exec_globals)
            if 'matmul' in exec_globals:
                rewards.append(1.0)
            else:
                rewards.append(0.0)
        except Exception:
            rewards.append(0.0)
    return rewards

def anti_cheating_reward(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        # AST check to prevent numpy or torch imports
        tree = ast.parse(completion)
        cheating = False
        for node in ast.walk(tree):
            if isinstance(node, ast.Import) or isinstance(node, ast.ImportFrom):
                cheating = True
        rewards.append(0.0 if cheating else 1.0)
    return rewards